In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel
import torch
import numpy as np
import pickle
from tqdm import tqdm

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_path = 'saved_models/graph/graph_model.pt'
bilstm_model_path = 'saved_models/bilstm/bilstm_model.pt'

In [3]:
graph_model = load_GraphModel(graph_model_path, device)
bilstm_model = load_BiLSTMModel(bilstm_model_path, device)

In [4]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

In [5]:
for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading hook
loading nott
loading wiki


In [6]:
print(datasets['gjt']['dataset'][0]['graph_ready_object'].bar_objects)

[<graph_utils.Bar object at 0x762fb00b75d0>, <graph_utils.Bar object at 0x762fb00d31d0>, <graph_utils.Bar object at 0x762fb013a6d0>, <graph_utils.Bar object at 0x762fb230ae10>, <graph_utils.Bar object at 0x762fb230a910>, <graph_utils.Bar object at 0x762fb00c3d10>, <graph_utils.Bar object at 0x762fa5f11110>, <graph_utils.Bar object at 0x762fb00c2390>, <graph_utils.Bar object at 0x762fb00d3790>, <graph_utils.Bar object at 0x76300d15b5d0>, <graph_utils.Bar object at 0x762fb00b7210>, <graph_utils.Bar object at 0x762fb00d1ad0>, <graph_utils.Bar object at 0x762fb00d9a90>, <graph_utils.Bar object at 0x762fa5f11a50>, <graph_utils.Bar object at 0x762fb00d27d0>, <graph_utils.Bar object at 0x762fa5f12ed0>]


In [7]:
print(datasets['gjt']['dataset'][0]['graph_ready_object'].segment_graph)

None


In [18]:
datasets['gjt']['dataset'][0]['graph_ready_object'].make_graph_of_segment(0,2)
datasets['gjt']['dataset'][0]['graph_ready_object'].make_bilstm_seq_of_segment(0,2)

In [19]:
print(datasets['gjt']['dataset'][0]['graph_ready_object'].__dict__)

{'bar_objects': [<graph_utils.Bar object at 0x762fb00b75d0>, <graph_utils.Bar object at 0x762fb00d31d0>, <graph_utils.Bar object at 0x762fb013a6d0>, <graph_utils.Bar object at 0x762fb230ae10>, <graph_utils.Bar object at 0x762fb230a910>, <graph_utils.Bar object at 0x762fb00c3d10>, <graph_utils.Bar object at 0x762fa5f11110>, <graph_utils.Bar object at 0x762fb00c2390>, <graph_utils.Bar object at 0x762fb00d3790>, <graph_utils.Bar object at 0x76300d15b5d0>, <graph_utils.Bar object at 0x762fb00b7210>, <graph_utils.Bar object at 0x762fb00d1ad0>, <graph_utils.Bar object at 0x762fb00d9a90>, <graph_utils.Bar object at 0x762fa5f11a50>, <graph_utils.Bar object at 0x762fb00d27d0>, <graph_utils.Bar object at 0x762fa5f12ed0>], 'num_bars': 16, 'segment_bar_end': 2, 'segment_bar_start': 0, 'segment_graph': HeteroData(
  pitch={ x=[12, 12] },
  event={ x=[2, 1] },
  (pitch, participates, event)={
    edge_index=[2, 11],
    edge_attr=[11, 8],
  },
  (event, next, event)={
    edge_index=[2, 1],
    edge

In [25]:
print(datasets['gjt']['dataset'][0]['graph_ready_object'].segment_bilstm.shape)

torch.Size([2, 24])


In [38]:
graph_embeddings = []
bilstm_embeddings = []
metadata = []

for k,v in datasets.items():
    for i, d in tqdm(enumerate(v['dataset'])):
        g = d['graph_ready_object']
        bar_objects = g.bar_objects
        for bar_start in range(len(bar_objects)-2):
            bar_end = bar_start + 2
            # graph
            g.make_graph_of_segment(bar_start, bar_end)
            z_graph = graph_model(g.segment_graph.to(device))
            graph_embeddings.append(
                z_graph.detach().cpu().numpy()
            )
            # bilstm
            g.make_bilstm_seq_of_segment(bar_start, bar_end)
            segment_bilstm = g.segment_bilstm.unsqueeze(0)
            lengths = torch.tensor(segment_bilstm.shape[1], dtype=int).unsqueeze(0)
            z_bilstm = bilstm_model(segment_bilstm.to(device), lengths.to(device))
            bilstm_embeddings.append(
                z_bilstm.detach().cpu().numpy()
            )

graph_embeddings = np.vstack(graph_embeddings)
bilstm_embeddings = np.vstack(bilstm_embeddings)

0it [00:00, ?it/s]

28it [00:00, 80.32it/s]
758it [00:05, 150.61it/s]
45it [00:00, 76.71it/s]
222it [00:02, 82.43it/s]


In [39]:
print(graph_embeddings.shape)
print(bilstm_embeddings.shape)

(9842, 128)
(9842, 128)


In [ ]:
# ['graph_ready_object'].make_graph_of_segment(bar_start, bar_end)
# ['graph_ready_object'].make_bilstm_seq_of_segment(bar_start, bar_end)